# Train EGNN Model

In [1]:
from IPython.display import display
import os

if "SSH_CONNECTION" in os.environ:
    display("Running via SSH")
else:
    display("Running locally")
    
import sys
import os

path = os.path.join('..', '.')
if path not in sys.path:
    sys.path.append(os.path.abspath(path))

import random

import numpy as np
import pandas as pd
import pickle as pkl

import torch
from torch_geometric.data import Data

import warnings
warnings.filterwarnings('ignore')

from src import run_model, protein_graph, evaluation


torch.cuda.is_available()

'Running via SSH'

True

In [2]:
# dataset_path = 'datasets/x18_exp_l2_graph_dict.pkl'
# dataset_path = 'datasets/clustered_2_graph_dict.pkl'
dataset_path = 'datasets/codon_graph_dict.pkl'
with open(dataset_path, 'rb') as f:
    graph_dict = pkl.load(f)
    
print(len(graph_dict['train']) + len(graph_dict['test']))

664


In [3]:
# Ensure EGNN has node coordinates on each Data object.
def _ensure_pos(graph_obj):
    data = graph_obj.dataset[0]
    if getattr(data, "pos", None) is not None:
        return False
    if hasattr(graph_obj, "node_positions"):
        pos = graph_obj.node_positions
    elif hasattr(graph_obj, "nodes"):
        pos = torch.tensor(
            graph_obj.nodes.center_of_mass(compound="residues"), dtype=torch.float
        )
    else:
        raise AttributeError("Graph object has no node positions.")
    data.pos = pos
    return True

updated = 0
total = 0
for split in ["train", "test"]:
    for sample in graph_dict[split]:
        total += 1
        graph_obj = graph_dict[split][sample]["graph"]
        if _ensure_pos(graph_obj):
            updated += 1

print(f"Added pos to {updated}/{total} graphs")

Added pos to 664/664 graphs


In [4]:
# remove meta features
for split in ["train", "test"]:
    for sample in graph_dict[split]:
        graph_obj = graph_dict[split][sample]["graph"]
        graph_obj.dataset[0].x = graph_obj.dataset[0].x[:, :-4]
        assert graph_obj.dataset[0].x.shape[1] == 14

### Train

- Cutoff distance = 12
- Dropout = 0.6
- Edge weights = "exp"
- Edge weight lambda = 2
- Hidden channels = 256
- Learning rate = 4.5e-5
- Weight decay = 5e-6
- EGNN irreps_sh = "0e+1o"
- EGNN num_basis = 10
- EGNN max_radius = 12

In [8]:
seed = 42
np.random.seed(seed)
random.seed(seed)

# logging params (only used for wandb metrics)
n_samples = len(graph_dict['train']) + len(graph_dict['test'])
cutoff_distance = 12

# egnn params
# num_node_features = 18
num_node_features = 14
batch_size = 128
hidden_channels = 64
dropout = 0.6

irreps_sh = "0e+1o"
num_basis = 10
max_radius = 6

edge_weight_func = "exp"
edge_weight_lambda = 2

learning_rate = 4e-1
wd = 5e-6
# epochs = 1119
epochs = 500

In [9]:
project = 'pnca-clustered-split'
run_name = 'codon-split-long'
# run_name = 'clustered_2-split-1'

In [10]:
model = run_model.pnca_EGNN_vary_graph(
            self_loops = False,
            cutoff_distance = cutoff_distance,
            edge_weight_func = edge_weight_func,
            batch_size = batch_size,
            num_node_features = num_node_features,
            hidden_channels = hidden_channels,
            learning_rate = learning_rate,
            wd = wd,
            dropout = dropout,
            lr_scheduling=False,
            epochs = epochs,
            graph_dict= graph_dict,
            normalise_ews=True,
            lambda_param= edge_weight_lambda,
            irreps_sh = irreps_sh,
            num_basis = num_basis,
            max_radius = max_radius,
            early_stop=False,
            # save_path= f'../saved_models/{project}/{run_name}',
            wandb_params={
              'use_wandb': False, 
              'wandb_project': f'{project}', 
              'wandb_name': f'{run_name}',
              'n_samples': n_samples,
              'sweep': False
              }
        )

Using CUDA
Epoch: 000, Train Acc: 0.5086, Test Acc: 0.5650, Train Loss: 47.0970, Test Loss: 43.3787
Epoch: 010, Train Acc: 0.5086, Test Acc: 0.5650, Train Loss: 0.7034, Test Loss: 0.6894
Epoch: 020, Train Acc: 0.5086, Test Acc: 0.5650, Train Loss: 0.6933, Test Loss: 0.6890
Epoch: 030, Train Acc: 0.5086, Test Acc: 0.5650, Train Loss: 0.6941, Test Loss: 0.6899
Epoch: 040, Train Acc: 0.5086, Test Acc: 0.5650, Train Loss: 0.6932, Test Loss: 0.6917
Epoch: 050, Train Acc: 0.5086, Test Acc: 0.5650, Train Loss: 0.6974, Test Loss: 0.6880
Epoch: 060, Train Acc: 0.5086, Test Acc: 0.5650, Train Loss: 0.6939, Test Loss: 0.6892
Epoch: 070, Train Acc: 0.4914, Test Acc: 0.4350, Train Loss: 0.6958, Test Loss: 0.6989
Epoch: 080, Train Acc: 0.4914, Test Acc: 0.4350, Train Loss: 0.6952, Test Loss: 0.6993
Epoch: 090, Train Acc: 0.4914, Test Acc: 0.4350, Train Loss: 0.6933, Test Loss: 0.6940
Epoch: 100, Train Acc: 0.4914, Test Acc: 0.4350, Train Loss: 0.6935, Test Loss: 0.6959
Epoch: 110, Train Acc: 0.5086,

KeyboardInterrupt: 